In [ ]:
import numpy as np
import os
import pandas as pd
import pandas_gbq
import matplotlib.pyplot as plt

In [ ]:
# Read in the sample lists
ufc_ids = pd.read_csv("/home/jupyter/ufc_subjects.aou.list", header=None,dtype=str)[0].tolist()
ufc_ids_exclude = pd.read_csv("/home/jupyter/ufc_subjects.aou.pgv_carriers_in_154_cpgs.list", header=None, dtype=str)[0].tolist()
ufc_ids = [x for x in ufc_ids if x not in set(ufc_ids_exclude)]

v7_ids = pd.read_csv("/home/jupyter/v7.subjects", header=None,dtype=str)[0].tolist()
#v9_ids = pd.read_csv("/home/jupyter/v9.subjects", header=None,dtype=str)[0].tolist()

# Convert to sets for comparison
ufc_set = set(ufc_ids)
v7_set = set(v7_ids)
#v9_set = set(v9_ids)

# Report
print(f"UFC samples: {len(ufc_set):,}")
print(f"CDRv7 samples: {len(v7_set):,}")
#print(f"CDRv9 samples: {len(v9_set):,}")
print()
print(f"UFC samples in CDRv7: {len(ufc_set & v7_set):,}")
# print(f"UFC samples in CDRv9: {len(ufc_set & v9_set):,}")
# print()
# print(f"CDRv7 samples not in CDRv9: {len(v7_set - v9_set):,}")

In [ ]:
CDRv7_query_code = "C2022Q4R13"
CDRv9_query_code = "C2025Q4R6"

In [ ]:
def get_peoples_cancer_family_history(data_version: str, cancer_types=["bladder","blood or soft tissue","brain","breast","bone","cervical","head and neck","stomach","other cancer","skin","pancreatic","colon","endocrine","esophagael","eye","kidney","lung","ovarian","thyroid","endometrial","prostate"]):
    query = f"""
    SELECT person_id, answer
    FROM `wb-silky-artichoke-2408.{data_version}.cb_review_survey`
    WHERE LOWER(answer) LIKE "%cancer%" AND LOWER(answer) LIKE "%including%"
    """

    df = pd.read_gbq(query)
    df["person_id"] = df["person_id"].astype(str)
    df = df[~df["answer"].str.contains("Self|non-cancer", na=False)]
    df["family_member"] = df["answer"].str.split("-", n=1).str[1].str.strip()

    # First Degree Relatives
    df_fdr = df[~df["family_member"].isin(["Grandparent", "Grandchild"])]
          
    df_num_fam_count = df[["person_id","family_member"]].drop_duplicates().groupby("person_id").size().reset_index(name="num_relatives_w_cancer")
    df_num_fdr_count = df_fdr[["person_id","family_member"]].drop_duplicates().groupby("person_id").size().reset_index(name="num_fdrs_w_cancer")

    dfs = []
    for cancer in cancer_types:
        df_cancer = df[df["answer"].str.contains(cancer, case=False, na=False)]
        df_cancer_fdr = df_cancer[~df_cancer["family_member"].isin(["Grandparent","Grandchild"])]

        # All denotated male relatives are also FDRs (e.g., GrandFather or Grandson aren't valid options in the survey)
        if cancer == "breast":
            male_cancer = df_cancer[df_cancer["family_member"].isin(["Father", "Son"])]
            male_cancer_fdr = male_cancer[~male_cancer["family_member"].isin(["Grandparent","Grandchild"])]
    
            df_cancer = df_cancer[~df_cancer["family_member"].isin(["Father", "Son"])]
            df_cancer_fdr = df_cancer_fdr[~df_cancer_fdr["family_member"].isin(["Father", "Son"])]

            # Make specific columns for male breast cancer
            male_relatives = male_cancer[["person_id","family_member"]].drop_duplicates().groupby("person_id").size().reset_index(name="male_relatives_w_breast_cancer")
            male_fdrs = male_cancer_fdr[["person_id","family_member"]].drop_duplicates().groupby("person_id").size().reset_index(name="male_fdrs_w_breast_cancer")
    
            dfs.append(male_relatives.merge(male_fdrs, on="person_id", how="outer"))
        
        relatives = df_cancer[["person_id","family_member"]].drop_duplicates().groupby("person_id").size().reset_index(name=f"relatives_w_{cancer}_cancer".replace(" ","_"))
        fdrs = df_cancer_fdr[["person_id","family_member"]].drop_duplicates().groupby("person_id").size().reset_index(name=f"fdrs_w_{cancer}_cancer".replace(" ","_"))

        dfs.append(relatives.merge(fdrs, on="person_id", how="outer"))

    result = df_num_fam_count.merge(df_num_fdr_count, on="person_id", how="outer")
    for df_cancer in dfs:
        result = result.merge(df_cancer, on="person_id", how="outer")

    count_cols = [col for col in result.columns if col != "person_id"]
    result[count_cols] = result[count_cols].fillna(0).astype(int)

    return result


df_num_fam_count_v7 = get_peoples_cancer_family_history(data_version = "C2022Q4R13")
#df_num_fam_count_v9 = get_peoples_cancer_family_history(data_version = "C2025Q4R6")

In [ ]:
query = """
SELECT
    person_id,
    MAX(start_datetime) AS last_survey_datetime
FROM `wb-silky-artichoke-2408.C2022Q4R13.cb_review_survey`
WHERE LOWER(answer) LIKE "%cancer%"
    AND LOWER(answer) LIKE "%including%"
GROUP BY person_id
ORDER BY last_survey_datetime DESC
"""

df_last_survey = pd.read_gbq(query)
df_last_survey["person_id"] = df_last_survey["person_id"].astype(str)
df_last_survey["last_survey_datetime"] = pd.to_datetime(df_last_survey["last_survey_datetime"])

In [ ]:
def get_person_demographics(data_version:str):
    query = f"""
    SELECT
        st.person_id,
        st.birth_datetime,
        dt_race.concept_name AS race,
        dt_sex.concept_name AS sex_at_birth
    FROM
        `wb-silky-artichoke-2408.{data_version}`.person AS st
    LEFT JOIN
        `wb-silky-artichoke-2408.{data_version}`.concept AS dt_race
        ON dt_race.concept_id = st.race_concept_id
    LEFT JOIN
        `wb-silky-artichoke-2408.{data_version}`.concept AS dt_sex
        ON dt_sex.concept_id = st.sex_at_birth_concept_id
    """

    df = pd.read_gbq(query)
    df["person_id"] = df["person_id"].astype(str)
    return df

df_demographics_v7 = get_person_demographics(data_version = "C2022Q4R13")
#df_demographics_v9 = get_person_demographics(data_version = "C2025Q4R6")

In [ ]:
def get_data_availability(data_version:str):
    query = f"""
    SELECT
        p.id AS person_id,
        CAST(p.has_ehr_data AS INT64) AS has_ehr,
        CAST(p.has_whole_genome_variant AS INT64) AS has_wgs,
        s.answer AS knowledge_of_fhx,
        DATE(s.start_datetime) AS fhx_survey_start_date
    FROM `wb-silky-artichoke-2408.{data_version}_index_061026.T_ENT_person` p
    LEFT JOIN `wb-silky-artichoke-2408.{data_version}.cb_review_survey` s
        ON p.id = s.person_id
        AND LOWER(s.question) LIKE "%how much do you know about illnesses or health problems for your parents, grandparents, brothers, sisters, and/or children?%"
    """
    df = pd.read_gbq(query)
    df["person_id"] = df["person_id"].astype(str)
    df = df.rename(columns={"has_ehr": "has_ehr_v9"})
    df["knowledge_of_fhx"] = df["knowledge_of_fhx"].str.lower()
    df.loc[df["knowledge_of_fhx"].str.contains("none at all", na=False), "knowledge_of_fhx"] = 0
    df.loc[df["knowledge_of_fhx"].str.contains("some", na=False), "knowledge_of_fhx"] = 1
    df.loc[df["knowledge_of_fhx"].str.contains("a lot", na=False), "knowledge_of_fhx"] = 2
    return df

#df_data_modalities_v7 = get_data_availability(data_version = "C2022Q4R13")
#df_data_modalities_v9 = get_data_availability(data_version = "C2025Q4R6")

In [ ]:
def get_knowledge_of_fhx_v7(data_version="C2022Q4R13"):
    query = f"""
    SELECT
        person_id,
        answer AS knowledge_of_fhx,
        DATE(start_datetime) AS fhx_survey_start_date
    FROM `wb-silky-artichoke-2408.{data_version}.cb_review_survey`
    WHERE LOWER(question) LIKE "%how much do you know about illnesses or health problems for your parents, grandparents, brothers, sisters, and/or children?%"
    """
    df = pd.read_gbq(query)
    df["person_id"] = df["person_id"].astype(str)
    df["knowledge_of_fhx"] = df["knowledge_of_fhx"].str.lower()
    df.loc[df["knowledge_of_fhx"].str.contains("none at all", na=False), "knowledge_of_fhx"] = 0
    df.loc[df["knowledge_of_fhx"].str.contains("some", na=False), "knowledge_of_fhx"] = 1
    df.loc[df["knowledge_of_fhx"].str.contains("a lot", na=False), "knowledge_of_fhx"] = 2
    return df

df_knowledge_fhx_v7 = get_knowledge_of_fhx_v7()

In [ ]:
# def get_last_clinical_followup(data_version: str):
#     query = f"""
#     SELECT person_id, MAX(clinical_date) AS last_clinical_followup
#     FROM (
#         SELECT person_id, procedure_date AS clinical_date
#         FROM `wb-silky-artichoke-2408.{data_version}.procedure_occurrence`
#         UNION ALL
#         SELECT person_id, visit_start_date AS clinical_date
#         FROM `wb-silky-artichoke-2408.{data_version}.visit_occurrence`
#         UNION ALL
#         SELECT person_id, condition_start_date AS clinical_date
#         FROM `wb-silky-artichoke-2408.{data_version}.condition_occurrence`
#     )
#     GROUP BY person_id
#     """

#     df = pd.read_gbq(query)
#     df["person_id"] = df["person_id"].astype(str)
#     return df

# df_last_clinical_followup_v7 = get_last_clinical_followup(data_version=CDRv7_query_code)

In [ ]:
def get_smoking_frequency(data_version: str):
    query = f"""
    SELECT person_id, question, answer
    FROM `wb-silky-artichoke-2408.{data_version}.cb_review_survey`
    WHERE LOWER(question) LIKE "%smoking: smoke frequency%"
    ORDER BY question
    """
    
    df = pd.read_gbq(query)
    df["person_id"] = df["person_id"].astype(str)
    df = df.rename(columns={"answer": "smoking_frequency"})
    df = df.drop(columns=["question"])

    smoking_map = {"Not At All": 0, "Some Days": 1, "Every Day": 2}
    df["smoking_frequency_scaled"] = df["smoking_frequency"].apply(lambda x: "NA" if pd.isna(x) or "PMI" in str(x) else next((v for k, v in smoking_map.items() if k.lower() in str(x).lower()), pd.NA))
    return df

df_smoking_frequency_v7 = get_smoking_frequency(data_version=CDRv7_query_code)
#df_smoking_frequency_v9 = get_smoking_frequency(data_version="C2025Q4R6")

In [ ]:
# def get_death(data_version: str):
#     query = f"""
#     SELECT person_id, death_date
#     FROM `wb-silky-artichoke-2408.{data_version}.death`
#     """

#     df = pd.read_gbq(query)
#     df["person_id"] = df["person_id"].astype(str)
#     return df

# df_death_v9 = get_death(data_version="C2025Q4R6")

In [ ]:
def get_alcohol_frequency(data_version: str):
    query = f"""
    SELECT person_id, question, answer
    FROM `wb-silky-artichoke-2408.{data_version}.cb_review_survey`
    WHERE LOWER(question) LIKE "%alcohol: drink frequency past year%"
    ORDER BY question
    """

    df = pd.read_gbq(query)
    df["person_id"] = df["person_id"].astype(str)
    df = df.rename(columns={"answer": "alcohol_frequency"})
    df = df.drop(columns=["question"])

    alcohol_map = {"Never": 0, "Monthly Or Less": 1, "2 to 4 Per Month": 2, "2 to 3 Per Week": 3, "4 or More Per Week": 4}
    df["alcohol_frequency_scaled"] = df["alcohol_frequency"].apply(lambda x: "NA" if pd.isna(x) or "PMI" in str(x) else next((v for k, v in alcohol_map.items() if k.lower() in str(x).lower()), pd.NA))
    return df
    
df_alcohol_frequency_v7 = get_alcohol_frequency(data_version=CDRv7_query_code)
#df_alcohol_frequency_v9 = get_alcohol_frequency(data_version="C2025Q4R6")

In [ ]:
def return_query(omop_code: int,data_version: str):
    return f"""
    SELECT
        co.person_id,
        co.condition_concept_id,
        MIN(co.condition_start_date) AS condition_start_date
    FROM `wb-silky-artichoke-2408.{data_version}.condition_occurrence` co
    JOIN `wb-silky-artichoke-2408.{data_version}.concept_ancestor` ca
        ON co.condition_concept_id = ca.descendant_concept_id
    WHERE ca.ancestor_concept_id = {omop_code}
        AND ca.min_levels_of_separation >= 0
    GROUP BY co.person_id, co.condition_concept_id
    """

def get_descendant_codes(omop_codes,data_version: str):
    codes = ", ".join(map(str, omop_codes))
    query = f"""
    SELECT DISTINCT descendant_concept_id
    FROM `wb-silky-artichoke-2408.{data_version}.concept_ancestor`
    WHERE ancestor_concept_id IN ({codes})
    """
    
    df = pd.read_gbq(query)
    return df["descendant_concept_id"].tolist()

# Define the 12 cancer types and the corresponding codes we will use 443392: "pancancer"
omop_codes = {
    443392: "pancancer",
    4112853: "breast",
    4178976: "thyroid",
    197508: "bladder",
    4038838: "nhl",
    4181351: "ovarian",
    197230: "endometrial",
    196653: "kidney",
    4162276: "melanoma",
    4163261: "prostate",
    37168850: "colon",
    443388: "lung",
    317510: "leukemia",
    4180793: "pancreas",

}

#cutaneous melanoma, non-medullary thyroid cancer, corpus uteri cancer
# no skin cancer of breast, and no uterine cervix cancers, only cutanoues melanomas when possible.
omop_codes_exclude = {
	"breast": [4244051,4247348],
	"endometrial": [196359,441805,198984,4092515],
	"melanoma": [4116085,36712732, 4170619],
    "kidney":[195480,4181357,601140],
    "colon":[1244634,44504337,35624314],
    "lung":[257513],
    "nhl":[4216139,437233,4082311,4149840],
    "ovarian":[196738,4281030,4281027]
}

breast_exclude = omop_codes_exclude["breast"]
endometrial_exclude = omop_codes_exclude["endometrial"]
melanoma_exclude = omop_codes_exclude["melanoma"]
kidney_exclude = omop_codes_exclude["kidney"]
colon_exclude = omop_codes_exclude["colon"]
lung_exclude = omop_codes_exclude["lung"]
nhl_exclude = omop_codes_exclude["nhl"]
ovarian_exclude = omop_codes_exclude["ovarian"]

In [ ]:
# def get_excluded_person_ids(omop_codes: dict, data_version: str):
#     codes = [code for code_list in omop_codes.values() for code in code_list]
#     codes_str = ", ".join(map(str, codes))

#     query = f"""
#     SELECT DISTINCT co.person_id
#     FROM `wb-silky-artichoke-2408.{data_version}.condition_occurrence` co
#     JOIN `wb-silky-artichoke-2408.{data_version}.concept_ancestor` ca
#         ON co.condition_concept_id = ca.descendant_concept_id
#     WHERE ca.ancestor_concept_id IN ({codes_str})
#     """

#     df = pd.read_gbq(query)
#     return set(df["person_id"].astype(str))

In [ ]:
def curate_cancer_diagnoses(omop_code):
    cancer_name = omop_codes[omop_code]
    
    if cancer_name == "breast":
        df_part1 = pd.read_gbq(return_query(omop_code, data_version=CDRv7_query_code))
        df_part2 = pd.read_gbq(return_query(137809, data_version=CDRv7_query_code))
        df = pd.concat([df_part1, df_part2], ignore_index=True)
    elif cancer_name == "colon":
        df_part1 = pd.read_gbq(return_query(omop_code, data_version=CDRv7_query_code))
        df_part2 = pd.read_gbq(return_query(36683531, data_version=CDRv7_query_code))
        df = pd.concat([df_part1, df_part2], ignore_index=True)
    elif cancer_name == "endometrial":
        df_part1 = pd.read_gbq(return_query(omop_code, data_version=CDRv7_query_code))
        df_part2 = pd.read_gbq(return_query(35206248, data_version=CDRv7_query_code))
        df = pd.concat([df_part1, df_part2], ignore_index=True)
    else:
        df = pd.read_gbq(return_query(omop_code, data_version=CDRv7_query_code))
        
    df["person_id"] = df["person_id"].astype(str)
    df["condition_start_date"] = pd.to_datetime(df["condition_start_date"])

    exclusion_map = {
        "breast": breast_exclude,
        "endometrial": endometrial_exclude,
        "melanoma": melanoma_exclude,
        "kidney": kidney_exclude,
        "colon": colon_exclude,
        "lung": lung_exclude,
        "nhl": nhl_exclude,
        "ovarian": ovarian_exclude
    }

    if cancer_name in exclusion_map:
        excluded_person_ids = set(df.loc[df["condition_concept_id"].isin(exclusion_map[cancer_name]),"person_id"])

        # Remove Kidney cancer individuals who have both renal pelvis cancer and bladder cancer
        if cancer_name == "kidney":
            bladder_ids = set(pd.read_gbq(return_query(197508, data_version=CDRv9_query_code))["person_id"].astype(str))
            excluded_person_ids &= bladder_ids
            
        df = df[~df["person_id"].isin(excluded_person_ids)]

    df = df.sort_values("condition_start_date",ascending=True).drop_duplicates("person_id", keep="first")
        
    df[f"{cancer_name}_cancer"] = pd.Series(1, index=df.index, dtype=int)
    df[f"{cancer_name}_diagnosis_date"] = df["condition_start_date"]
    df[f"{cancer_name}_concept_id"] = df["condition_concept_id"]
    df = df.drop(columns=["condition_concept_id","condition_start_date"])
    return df

In [ ]:
dfs = []
for omop_code in omop_codes:
    dfs.append(curate_cancer_diagnoses(omop_code))

results = dfs[0]

for df in dfs[1:]:
    results = results.merge(df,on="person_id",how="outer")

In [ ]:
df_merged_v7 = results.merge(df_num_fam_count_v7, on="person_id",how="outer")
df_merged_v7 = pd.merge(df_merged_v7, df_demographics_v7, on="person_id", how="outer")
df_merged_v7 = pd.merge(df_merged_v7, df_smoking_frequency_v7, on="person_id", how="outer")
df_merged_v7 = pd.merge(df_merged_v7, df_alcohol_frequency_v7, on="person_id", how="outer")
df_merged_v7 = df_merged_v7[df_merged_v7['person_id'].isin(v7_set)]
    
# Impute those with no reported FDR in v7/9with cancer as having 0 FDR cancers
df_merged_v7["num_fdrs_w_cancer"] = df_merged_v7["num_fdrs_w_cancer"].fillna(0).astype(int)

# Make a new column for male breast cancer
df_merged_v7["male_breast_cancer"] = ((df_merged_v7["sex_at_birth"] == "Male") & (df_merged_v7["breast_cancer"] == 1)).astype(int)

# ensure that sex_at_birth is 'Female' for those with breast_cancer, endometrial_cancer, or ovarian_cancer == 1, similalry 'Male' with prostate_cancer == 1
df_merged_v7.loc[df_merged_v7["sex_at_birth"] == "Male", ["breast_cancer", "endometrial_cancer", "ovarian_cancer"]] = 0
df_merged_v7.loc[df_merged_v7["sex_at_birth"] == "Female", "prostate_cancer"] = 0


In [ ]:
def correct_pancancer_flags(df_merged_v7, data_version="C2022Q4R13"):
    query = f"""
    SELECT DISTINCT CAST(co.person_id AS STRING) AS person_id
    FROM `wb-silky-artichoke-2408.{data_version}.condition_occurrence` co
    JOIN `wb-silky-artichoke-2408.{data_version}.concept` c
        ON co.condition_concept_id = c.concept_id
    WHERE LOWER(c.concept_name) LIKE '%malignant neoplas%'
    """

    cancer_ids = set(pd.read_gbq(query)["person_id"].astype(str))
    df_merged_v7["person_id"] = df_merged_v7["person_id"].astype(str)

    mask = df_merged_v7["person_id"].isin(cancer_ids)
    previously_unflagged = mask & df_merged_v7["pancancer_cancer"].fillna(0).eq(0)

    df_merged_v7.loc[mask, "pancancer_cancer"] = 1

    print(f"Individuals with matching diagnoses in df_merged_v7: {mask.sum():,}")
    print(f"Previously unflagged individuals corrected: {previously_unflagged.sum():,}")

    return df_merged_v7

df_merged_v7 = correct_pancancer_flags(df_merged_v7)

In [ ]:
def exclude_neoplastic_controls(df):
    data_version = "C2022Q4R13"
    query = f"""
    SELECT DISTINCT CAST(person_id AS STRING) AS person_id
    FROM `wb-silky-artichoke-2408.{data_version}.condition_occurrence`
    WHERE condition_concept_id IN (438112, 432582)
    """
    excluded_ids = set(pd.read_gbq(query)["person_id"].astype(str).str.strip())

    mask = df["person_id"].astype(str).str.strip().isin(excluded_ids) & df["pancancer_cancer"].fillna(0).eq(0)
    print(f"Excluded controls: {mask.sum()}")
    return df.loc[~mask].copy()

df_merged_v7 = exclude_neoplastic_controls(df_merged_v7)

In [ ]:
ufc_df = df_knowledge_fhx_v7[df_knowledge_fhx_v7["person_id"].isin(ufc_set)]
ufc_df = ufc_df.merge(df_merged_v7, how="inner", on="person_id")
#ufc_df = ufc_df.merge(results[["person_id", "pancancer_cancer"]], how="left", on="person_id")

plot_df = ufc_df.copy()
plot_df["knowledge_of_fhx"] = plot_df["knowledge_of_fhx"].fillna("No response")
plot_df["knowledge_of_fhx"] = plot_df["knowledge_of_fhx"].replace({"pmi: skip": "Skip", 0: "None at all", 1: "Some", 2: "A lot"})
plot_df["case_control"] = plot_df["pancancer_cancer"].map({1: "Case"}).fillna("Control")

order = ["No response", "Skip", "None at all", "Some", "A lot"]

plot_df["has_1_relative"] = (plot_df["num_relatives_w_cancer"] >= 1).astype(int)
plot_df["has_3_relative"] = (plot_df["num_relatives_w_cancer"] >= 3).astype(int)

summary_1 = plot_df.groupby(["knowledge_of_fhx", "case_control"])["has_1_relative"].mean().unstack().reindex(order) * 100
summary_3 = plot_df.groupby(["knowledge_of_fhx", "case_control"])["has_3_relative"].mean().unstack().reindex(order) * 100
counts = plot_df.groupby(["knowledge_of_fhx", "case_control"]).size().unstack().reindex(order)

x = np.arange(len(order))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

for ax, summary, title in zip(
    axes,
    [summary_1, summary_3],
    ["≥1 Relative with cancer", "≥3 Relatives with cancer"]
):
    bars_case = ax.bar(x - width/2, summary["Case"], width, label="Cases")
    bars_control = ax.bar(x + width/2, summary["Control"], width, label="Controls")

    ax.set_xlabel("Knowledge of family history")
    ax.set_title(title)
    ax.set_xticks(x)
    ax.set_xticklabels(order)
    ax.set_ylim(0, 100)

    for bars, group in [(bars_case, "Case"), (bars_control, "Control")]:
        for bar, category in zip(bars, order):
            value = summary.loc[category, group]
            n = counts.loc[category, group]
            if pd.notna(value):
                ax.text(
                    bar.get_x() + bar.get_width()/2,
                    value + 1,
                    f"{value:.1f}%\n(N = {int(n):,})",
                    ha="center",
                    va="bottom"
                )

axes[0].set_ylabel("% reporting familial cancer")
axes[0].legend()

fig.suptitle("Family Cancer History by Knowledge of Family History", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
n_cases = (plot_df["case_control"] == "Case").sum()
n_controls = (plot_df["case_control"] == "Control").sum()

n_cases_lt3 = ((plot_df["case_control"] == "Case") & (plot_df["num_relatives_w_cancer"].fillna(0) < 3)).sum()
n_controls_ge1 = ((plot_df["case_control"] == "Control") & (plot_df["num_relatives_w_cancer"].fillna(0) >= 1)).sum()

print(f"Cases with <3 Relatives with cancer: {n_cases_lt3:,} / {n_cases:,} ({n_cases_lt3 / n_cases * 100:.1f}%)")
print(f"Controls with ≥1 Relative with cancer: {n_controls_ge1:,} / {n_controls:,} ({n_controls_ge1 / n_controls * 100:.1f}%)")

In [ ]:
cancer_columns = [
    "breast_cancer",
    "prostate_cancer",
    "melanoma_cancer",
    "thyroid_cancer",
    "colon_cancer",
    "lung_cancer",
    "bladder_cancer",
    "kidney_cancer",
    "endometrial_cancer",
    "ovarian_cancer",
    "nhl_cancer",
    "leukemia_cancer"
]

case_df = ufc_df[(ufc_df["pancancer_cancer"] == 1) & (ufc_df["num_relatives_w_cancer"] >= 3)].copy()

has_any_cancer = case_df[cancer_columns].fillna(0).eq(1).any(axis=1)

n_cases = has_any_cancer.sum()
n_total_cases = len(case_df)

print(f"Cases with ≥1 specified cancer: {n_cases:,} / {n_total_cases:,} ({n_cases / n_total_cases * 100:.1f}%)")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

cancers = ["melanoma", "breast", "prostate", "lung", "colon", "thyroid", "nhl", "bladder", "ovarian", "kidney", "leukemia","endometrial"]
fdr_columns = {"melanoma": "fdrs_w_skin_cancer", "nhl": "fdrs_w_blood_or_soft_tissue_cancer",  "leukemia": "fdrs_w_blood_or_soft_tissue_cancer"}

fig, axes = plt.subplots(3, 4, figsize=(18, 12), constrained_layout=True)
axes = axes.flatten()

for i, cancer in enumerate(cancers):
    cancer_df = ufc_df[ufc_df[f"{cancer}_cancer"] == 1].copy()
    fdr_column = fdr_columns.get(cancer, f"fdrs_w_{cancer}_cancer")
    cancer_df[fdr_column] = cancer_df[fdr_column].fillna(0).astype(int)

    cancer_counts = pd.Series({
        "0": (cancer_df[fdr_column] == 0).sum(),
        "≥1": (cancer_df[fdr_column] >= 1).sum(),
        "≥2": (cancer_df[fdr_column] >= 2).sum()
    })

    total = len(cancer_df)
    percentages = cancer_counts / total * 100

    bars = axes[i].bar(cancer_counts.index, cancer_counts.values)

    axes[i].set_title(f"{cancer} (N = {total})")
    axes[i].set_xlabel("Number of Relatives")
    axes[i].set_ylabel("Number of individuals")
    axes[i].set_ylim(0, cancer_counts.max() * 1.25)

    for bar, count, percentage in zip(bars, cancer_counts.values, percentages.values):
        axes[i].text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + cancer_counts.max() * 0.02,
            f"{count:,} ({percentage:.1f}%)",
            ha="center",
            va="bottom"
        )

for i in range(len(cancers), len(axes)):
    axes[i].axis("off")

fig.suptitle("Number of First-Degree Relatives with the Same Cancer", fontsize=16)
plt.show()

In [ ]:
# # Save files locally, then copy to gsutil
# df_merged_v7.to_csv("CDRv7.metadata",sep='\t',index=False)
# !gsutil -m cp CDRv7.metadata gs://dataproc-staging-wb-cordial-diamond-9893/UFC_REFERENCE_FILES/

# ufc_df.to_csv("CDRv7.ufc_original.metadata",sep='\t',index=False)
# !gsutil -m cp CDRv7.ufc_original.metadata gs://dataproc-staging-wb-cordial-diamond-9893/UFC_REFERENCE_FILES/